# nnUNet v2 — 3D CT 分割训练 (NIfTI, 512×512×Z)

本 notebook 完整覆盖：
1. 环境安装
2. 路径配置
3. 数据集准备（将 `.nii` 转换为 nnUNet 格式）
4. 数据集完整性校验 + 预处理
5. 模型训练（3d_fullres）
6. 寻找最佳配置
7. 推理

**数据假设**
- 图像格式：`.nii` 或 `.nii.gz`
- 空间尺寸：`512 × 512 × Z`（Z 不固定）
- 单模态（CT），灰度值为 HU
- mask：整数标签，背景为 0，前景类别从 1 开始连续编号

## 1. 安装依赖

In [1]:
# 先确认 PyTorch（需与 CUDA 版本匹配），再安装 nnUNetv2
# 若已安装可跳过
import subprocess, sys

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print('[STDERR]', result.stderr)

# 检查 PyTorch
try:
    import torch
    print(f'PyTorch {torch.__version__}, CUDA available: {torch.cuda.is_available()}')
    if torch.cuda.is_available():
        print(f'GPU: {torch.cuda.get_device_name(0)}')
except ImportError:
    print('PyTorch 未安装，请先运行:')
    print('  pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121')

PyTorch 2.10.0+cpu, CUDA available: False


In [2]:
# 安装 nnunetv2（如已安装则跳过）
run(f'{sys.executable} -m pip install nnunetv2 -q')

# 可选：安装网络可视化工具
# run(f'{sys.executable} -m pip install --upgrade git+https://github.com/FabianIsensee/hiddenlayer.git -q')


[STDERR] /bin/sh: 1: /mnt/d/code/project: not found



## 2. 路径配置

nnUNet 依赖三个环境变量。修改下面的路径后，整个 notebook 均以此为基准。

In [3]:
import os
from pathlib import Path

# ============================================================
# 修改这里的路径
BASE_DIR        = Path('/mnt/d/code/project AI-MED/HNSCC/MAR6_nnunet')

RAW_DIR         = BASE_DIR / 'nnUNet_raw'
PREPROCESSED_DIR = BASE_DIR / 'nnUNet_preprocessed'
RESULTS_DIR     = BASE_DIR / 'nnUNet_results'

# 数据集编号和名称（Dataset{ID}_{NAME} 的格式）
DATASET_ID      = 100          # 任意三位数，避免与官方示例冲突
DATASET_NAME    = 'HNSCC_CT'  # 自定义名称，不含空格

# 原始 NIfTI 数据所在目录
# 期望结构：
#   SOURCE_IMAGES/case001.nii(.gz), case002.nii(.gz), ...
#   SOURCE_MASKS/ case001.nii(.gz), case002.nii(.gz), ...
SOURCE_IMAGES   = BASE_DIR / 'raw_data' / 'images'
SOURCE_MASKS    = BASE_DIR / 'raw_data' / 'masks'

# 前景类别标签（背景固定为 0）
LABELS = {
    'background': 0,
    'tumor':      1,   # 按实际类别修改
    # 'lymph_node': 2,
}
# ============================================================

# 设置环境变量（当前进程及子进程均可见）
os.environ['nnUNet_raw']         = str(RAW_DIR)
os.environ['nnUNet_preprocessed']= str(PREPROCESSED_DIR)
os.environ['nnUNet_results']     = str(RESULTS_DIR)

# 创建目录
for d in [RAW_DIR, PREPROCESSED_DIR, RESULTS_DIR, SOURCE_IMAGES, SOURCE_MASKS]:
    d.mkdir(parents=True, exist_ok=True)

DATASET_FOLDER = RAW_DIR / f'Dataset{DATASET_ID:03d}_{DATASET_NAME}'

print('nnUNet_raw         :', os.environ['nnUNet_raw'])
print('nnUNet_preprocessed:', os.environ['nnUNet_preprocessed'])
print('nnUNet_results     :', os.environ['nnUNet_results'])
print('Dataset folder     :', DATASET_FOLDER)

nnUNet_raw         : /mnt/d/code/project AI-MED/HNSCC/MAR6_nnunet/nnUNet_raw
nnUNet_preprocessed: /mnt/d/code/project AI-MED/HNSCC/MAR6_nnunet/nnUNet_preprocessed
nnUNet_results     : /mnt/d/code/project AI-MED/HNSCC/MAR6_nnunet/nnUNet_results
Dataset folder     : /mnt/d/code/project AI-MED/HNSCC/MAR6_nnunet/nnUNet_raw/Dataset100_HNSCC_CT


## 3. 数据集准备

将原始 `.nii/.nii.gz` 文件按 nnUNet 规范重命名，并生成 `dataset.json`。

**目标结构**
```
Dataset100_HNSCC_CT/
├── imagesTr/
│   ├── HNSCC_0001_0000.nii.gz   # _0000 = channel 0 (CT)
│   └── ...
├── imagesTs/          (可选，推理用)
├── labelsTr/
│   ├── HNSCC_0001.nii.gz
│   └── ...
└── dataset.json
```

In [4]:
import shutil
import json
import re

IMAGES_TR = DATASET_FOLDER / 'imagesTr'
IMAGES_TS = DATASET_FOLDER / 'imagesTs'
LABELS_TR = DATASET_FOLDER / 'labelsTr'

for d in [IMAGES_TR, IMAGES_TS, LABELS_TR]:
    d.mkdir(parents=True, exist_ok=True)

PREFIX = DATASET_NAME  # 文件名前缀

def get_sorted_niis(folder: Path):
    """返回目录下所有 .nii 和 .nii.gz 文件，按文件名排序。"""
    files = sorted(
        list(folder.glob('*.nii.gz')) + list(folder.glob('*.nii'))
    )
    return files

image_files = get_sorted_niis(SOURCE_IMAGES)
mask_files  = get_sorted_niis(SOURCE_MASKS)

print(f'找到图像：{len(image_files)} 个')
print(f'找到 mask：{len(mask_files)} 个')

assert len(image_files) == len(mask_files), \
    f'图像数量 ({len(image_files)}) 与 mask 数量 ({len(mask_files)}) 不匹配！'
assert len(image_files) > 0, f'在 {SOURCE_IMAGES} 下未找到任何 .nii/.nii.gz 文件'

找到图像：10 个
找到 mask：10 个


In [5]:
# 按 80/20 划分训练集与测试集（如不需要测试集可全放 imagesTr）
import math
import random

random.seed(42)
indices = list(range(len(image_files)))
random.shuffle(indices)

n_train = math.ceil(len(indices) * 0.8)
train_idx = sorted(indices[:n_train])
test_idx  = sorted(indices[n_train:])

print(f'训练集：{len(train_idx)} 例，测试集：{len(test_idx)} 例')

def copy_case(img_path: Path, mask_path: Path, case_id: int, split: str = 'train'):
    """将一对 image/mask 拷贝并重命名到 nnUNet 目录。"""
    suffix = '.nii.gz' if img_path.suffix == '.gz' else '.nii'
    case_name = f'{PREFIX}_{case_id:04d}'

    if split == 'train':
        dst_img  = IMAGES_TR / f'{case_name}_0000{suffix}'
        dst_mask = LABELS_TR / f'{case_name}{suffix}'
    else:
        dst_img  = IMAGES_TS / f'{case_name}_0000{suffix}'
        dst_mask = None  # 测试集不复制 mask

    if not dst_img.exists():
        shutil.copy2(img_path, dst_img)
    if dst_mask and not dst_mask.exists():
        shutil.copy2(mask_path, dst_mask)

    return case_name

training_cases = []
for cid, idx in enumerate(train_idx, start=1):
    name = copy_case(image_files[idx], mask_files[idx], cid, 'train')
    training_cases.append(name)

for cid, idx in enumerate(test_idx, start=n_train + 1):
    copy_case(image_files[idx], mask_files[idx], cid, 'test')

print('文件复制完毕。')
print('训练样例（前5）：', training_cases[:5])

训练集：8 例，测试集：2 例
文件复制完毕。
训练样例（前5）： ['HNSCC_CT_0001', 'HNSCC_CT_0002', 'HNSCC_CT_0003', 'HNSCC_CT_0004', 'HNSCC_CT_0005']


In [6]:
# 生成 dataset.json
dataset_json = {
    "channel_names": {
        "0": "CT"   # 单模态 CT；多模态可追加 "1": "PET" 等
    },
    "labels": LABELS,
    "numTraining": len(training_cases),
    "file_ending": ".nii.gz",
    "overwrite_image_reader_writer": "SimpleITKIO"  # 支持 .nii/.nii.gz
}

json_path = DATASET_FOLDER / 'dataset.json'
with open(json_path, 'w') as f:
    json.dump(dataset_json, f, indent=4)

print('dataset.json 内容：')
print(json.dumps(dataset_json, indent=4))

dataset.json 内容：
{
    "channel_names": {
        "0": "CT"
    },
    "labels": {
        "background": 0,
        "tumor": 1
    },
    "numTraining": 8,
    "file_ending": ".nii.gz",
    "overwrite_image_reader_writer": "SimpleITKIO"
}


## 4. 校验数据完整性

In [7]:
import subprocess, sys
from pathlib import Path as _Path

def nnunet_cmd(cmd: str, env_extra: dict | None = None):
    """运行 nnUNet CLI 命令，继承当前环境变量（含 nnUNet_* 路径）。
    
    自动将当前解释器所在的 bin 目录（venv 场景下的 .venv/bin）
    加入 PATH，确保 nnUNetv2_* 命令可被 shell 找到。
    """
    env = os.environ.copy()
    # 将 venv/conda bin 目录添加到 PATH 最前端
    venv_bin = str(_Path(sys.executable).parent)
    env['PATH'] = venv_bin + os.pathsep + env.get('PATH', '')
    if env_extra:
        env.update(env_extra)
    result = subprocess.run(
        cmd, shell=True, env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        raise RuntimeError(f'命令失败（exit {result.returncode}）：{cmd}')

# 校验数据集完整性 + 自动规划和预处理
nnunet_cmd(f'nnUNetv2_plan_and_preprocess -d {DATASET_ID} --verify_dataset_integrity -np 4')

Fingerprint extraction...
Dataset100_HNSCC_CT
Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> reader/writer

####################
verify_dataset_integrity Done. 
If you didn't see any error messages then your dataset is most likely OK!
####################

Experiment planning...

############################
INFO: You are using the old nnU-Net default planner. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Dropping 3d_lowres config because the image size difference to 3d_fullres is too small. 3d_fullres: [16. 64. 64.], 3d_lowres: [16, 64, 64]
2D U-Net configuration:
{'data_identifier': 'nnUNetPlans_2d', 'preprocessor_name': 'DefaultPreprocessor', 'batch_size': 6, 'patch_size': (np.int64(64), np.int64(64)), 'median_image_size_in_voxels': array([64., 64.]), 'spacing': array([1.5, 1.5]), 'normalization_schemes': ['CTN

## 5. 模型训练

对于 512×512×Z 的 CT，推荐使用 `3d_fullres` 配置。

默认进行 5 折交叉验证；可单独训练某一折（`FOLD` 设为 0-4 或 `all`）。

In [ ]:
# 训练配置
CONFIGURATION = '3d_fullres'  # 可选: '2d', '3d_lowres', '3d_cascade_fullres'

# TRAINER 可保持默认
TRAINER       = 'nnUNetTrainer'

# 在 Notebook 中通常先训练 fold=0 验证流程，完整实验再跑全部 5 折
FOLD          = 0   # 0-4 或 'all'

# GPU 设置（多卡时改为 '0,1' 等）
GPU_ID        = '0'

train_cmd = (
    f'CUDA_VISIBLE_DEVICES={GPU_ID} '
    f'nnUNetv2_train {DATASET_ID} {CONFIGURATION} {FOLD} '
    f'-tr {TRAINER} '
    f'--npz '  # 保存 softmax 输出，用于后续集成
)

print('即将执行：', train_cmd)
print('\n训练日志将实时输出……\n')
nnunet_cmd(train_cmd)

In [ ]:
# 可选：训练所有 5 折（建议用 shell 并行或 tmux 运行以节省时间）
# for fold in range(5):
#     cmd = (
#         f'CUDA_VISIBLE_DEVICES={GPU_ID} '
#         f'nnUNetv2_train {DATASET_ID} {CONFIGURATION} {fold} '
#         f'-tr {TRAINER} --npz'
#     )
#     nnunet_cmd(cmd)

## 6. 查找最佳配置

In [ ]:
# 训练完所有折后，自动选出最优配置及集成策略
nnunet_cmd(
    f'nnUNetv2_find_best_configuration {DATASET_ID} '
    f'-c {CONFIGURATION} '
    f'--disable_ensembling'  # 只选单配置；去掉此项可启用集成
)

## 7. 推理（Inference）

In [ ]:
# 推理输入/输出路径
INFER_INPUT  = DATASET_FOLDER / 'imagesTs'
INFER_OUTPUT = BASE_DIR / 'predictions'
INFER_OUTPUT.mkdir(parents=True, exist_ok=True)

# 自动选择最佳 checkpoint（best 或 final）
CHECKPOINT = 'checkpoint_best.pth'

predict_cmd = (
    f'CUDA_VISIBLE_DEVICES={GPU_ID} '
    f'nnUNetv2_predict '
    f'-i "{INFER_INPUT}" '
    f'-o "{INFER_OUTPUT}" '
    f'-d {DATASET_ID} '
    f'-c {CONFIGURATION} '
    f'-tr {TRAINER} '
    f'-chk {CHECKPOINT} '
    f'-f {FOLD} '
    f'--save_probabilities'  # 保存概率图（可选）
)

print('推理命令：', predict_cmd)
nnunet_cmd(predict_cmd)

## 8. 评估指标（可选）

若测试集有 ground truth，可用 nnUNet 内置评估工具计算 Dice 等指标。

In [ ]:
# 计算 Dice、HD95 等指标
# GROUND_TRUTH_DIR 需包含与预测文件同名的 mask
GROUND_TRUTH_DIR = LABELS_TR  # 示例：用训练集的 label 做验证（实际应用请替换）

eval_cmd = (
    f'nnUNetv2_evaluate_folder '
    f'-gt "{GROUND_TRUTH_DIR}" '
    f'-pred "{INFER_OUTPUT}" '
    f'-djfile "{DATASET_FOLDER}/dataset.json" '
    f'-pfile "{RESULTS_DIR}/Dataset{DATASET_ID:03d}_{DATASET_NAME}/{TRAINER}__{CONFIGURATION}/plans.json"'
)

nnunet_cmd(eval_cmd)

## 9. 实用工具：快速查看 NIfTI 文件信息

In [ ]:
try:
    import nibabel as nib
    import numpy as np
except ImportError:
    run(f'{sys.executable} -m pip install nibabel -q')
    import nibabel as nib
    import numpy as np

def inspect_nii(path: Path):
    img = nib.load(str(path))
    data = img.get_fdata()
    print(f'文件        : {path.name}')
    print(f'Shape       : {data.shape}  (H × W × Z)')
    print(f'Voxel spacing: {img.header.get_zooms()}')
    print(f'数据类型    : {data.dtype}')
    print(f'值域        : [{data.min():.1f}, {data.max():.1f}]')
    print(f'唯一标签    : {np.unique(data) if data.max() < 100 else "（连续值，非标签）"}')
    print()

# 展示前 3 个图像和 mask 的信息
for f in get_sorted_niis(SOURCE_IMAGES)[:3]:
    inspect_nii(f)

print('--- masks ---')
for f in get_sorted_niis(SOURCE_MASKS)[:3]:
    inspect_nii(f)

In [ ]:
# 可视化某张图像的中间切片（需安装 matplotlib）
try:
    import matplotlib.pyplot as plt
except ImportError:
    run(f'{sys.executable} -m pip install matplotlib -q')
    import matplotlib.pyplot as plt

def show_slice(img_path: Path, mask_path: Path | None = None, alpha: float = 0.4):
    img  = nib.load(str(img_path)).get_fdata()
    z    = img.shape[2] // 2  # 取中间层

    fig, axes = plt.subplots(1, 2 if mask_path else 1, figsize=(12, 5))
    if mask_path is None:
        axes = [axes]

    axes[0].imshow(img[:, :, z], cmap='gray', vmin=-200, vmax=300)
    axes[0].set_title(f'{img_path.name}  slice z={z}')
    axes[0].axis('off')

    if mask_path:
        mask = nib.load(str(mask_path)).get_fdata()
        axes[0].imshow(np.ma.masked_where(mask[:, :, z] == 0, mask[:, :, z]),
                       cmap='Reds', alpha=alpha)

        axes[1].imshow(mask[:, :, z], cmap='tab10', vmin=0, vmax=9)
        axes[1].set_title(f'{mask_path.name}  mask z={z}')
        axes[1].axis('off')

    plt.tight_layout()
    plt.show()

# 显示第一对图像/mask
imgs  = get_sorted_niis(SOURCE_IMAGES)
masks = get_sorted_niis(SOURCE_MASKS)
if imgs:
    show_slice(imgs[0], masks[0] if masks else None)

---
## 快速参考

| 步骤 | 命令 |
|------|------|
| 预处理 | `nnUNetv2_plan_and_preprocess -d {ID} --verify_dataset_integrity` |
| 训练（单折） | `nnUNetv2_train {ID} 3d_fullres 0 --npz` |
| 训练（全部） | 循环 fold 0-4 |
| 最佳配置 | `nnUNetv2_find_best_configuration {ID} -c 3d_fullres` |
| 推理 | `nnUNetv2_predict -i INPUT -o OUTPUT -d {ID} -c 3d_fullres -f 0` |
| 评估 | `nnUNetv2_evaluate_folder -gt GT_DIR -pred PRED_DIR -djfile dataset.json` |

**nnUNet 自动处理**：
- Patch size / batch size 根据 GPU 显存自动计算
- 512×512 平面 + 可变 Z 轴对 `3d_fullres` 完全兼容
- 数据增强、归一化（CT 使用 z-score per foreground voxel）自动配置